# Bài 3 · NumPy — mảng & tính toán vector hoá

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Tạo và mô tả được ndarray (`dtype`, `shape`), sinh dữ liệu ngẫu nhiên **có seed**.
2. Thay vòng lặp bằng **vector hoá**: công thức cả mảng, mặt nạ bool, `np.where`.
3. Dùng đúng `axis` khi tổng hợp ma trận; xử lý NaN chủ động; đo phân vị.
4. Tránh được bẫy **view vs copy**.

## 1. Vì sao NumPy? Tự đo lấy!

Cùng một việc — nhân 2 triệu số với 1.1 rồi cộng — làm hai kiểu:

In [ ]:
import time
import numpy as np

n = 2_000_000
xs_list = list(range(n))
xs_arr = np.arange(n)

t0 = time.perf_counter()
tong_1 = sum(x * 1.1 for x in xs_list)          # Python thuần
t_python = time.perf_counter() - t0

t0 = time.perf_counter()
tong_2 = (xs_arr * 1.1).sum()                   # NumPy
t_numpy = time.perf_counter() - t0

print(f"Python thuần: {t_python*1000:7.1f} ms")
print(f"NumPy       : {t_numpy*1000:7.1f} ms   (nhanh hơn ~{t_python/t_numpy:.0f} lần)")
assert abs(tong_1 - tong_2) < 1e-3              # cùng đáp số!

Cùng đáp số, tốc độ khác hẳn nhau — vì mảng NumPy là **một khối nhớ liền, cùng dtype**, vòng lặp
chạy trong C thay vì qua trình thông dịch Python.

## 2. Tạo mảng, dtype và reshape

In [ ]:
gia = np.array([1200, 950, 2400, 610])       # nghìn đồng/đêm
gia.dtype, gia.shape, gia.ndim

In [ ]:
# Ba cách tạo mảng hay dùng
print(np.arange(0, 10, 2))          # dãy cách đều
print(np.linspace(0, 1, 5))         # chia đều đoạn [0, 1] thành 5 mốc
print(np.zeros((2, 3)))             # khung rỗng

In [ ]:
# Sinh dữ liệu ngẫu nhiên CÓ SEED — chạy lại ra đúng số cũ (tái lập!)
rng = np.random.default_rng(seed=42)
gia_gia_lap = rng.normal(1000, 250, size=5).round()
gia_gia_lap

In [ ]:
# dtype quan trọng: CSV cho bạn chuỗi, phải astype trước khi tính
diem = np.array(["7.5", "8.0", "9.25"])
print(diem.dtype)                    # <U4: chuỗi unicode!
print(diem.astype(float).mean())

In [ ]:
# reshape: cùng dữ liệu, khác khung nhìn
doanh_thu_12_thang = np.arange(1, 13)
doanh_thu_12_thang.reshape(4, 3)     # 4 quý × 3 tháng

## 3. Vector hoá: công thức, mặt nạ, where

In [ ]:
# Công thức áp cho CẢ mảng — không for, không append
gia_sau_phi = gia * 1.15 + 50        # +15% phí, +50k dọn dẹp
gia_sau_phi

In [ ]:
# So sánh cho ra mảng bool — và hai phép thần thánh trên bool:
mask = gia > 1000
print(mask)
print("Số phòng > 1000k :", mask.sum())     # sum bool = ĐẾM
print("Tỷ lệ            :", mask.mean())    # mean bool = TỶ LỆ

In [ ]:
# Boolean indexing: lọc dữ liệu bẩn trong 1 dòng
gia_tho = np.array([1200, -5, 950, 0, 2400])
gia_tho[gia_tho > 0]

In [ ]:
# Ghép điều kiện: & | ~ với ngoặc quanh từng vế (KHÔNG dùng and/or)
hop_le = (gia_tho > 0) & (gia_tho < 2000)
gia_tho[hop_le]

In [ ]:
# np.where: if-else cho cả mảng
np.where(gia > 1000, "cao", "bình dân")

## 4. Broadcasting: mảng nhỏ tự "kéo giãn"

Ma trận giá 4 thành phố × 3 tháng, cộng phụ phí theo tháng (vector 3 phần tử):

In [ ]:
gia_4tp_3thang = np.array([[10, 12, 11],
                           [20, 21, 24],
                           [30, 33, 31],
                           [40, 44, 42]])
phu_phi_thang = np.array([1, 2, 3])

gia_4tp_3thang + phu_phi_thang       # vector tự áp lên TỪNG hàng

Quy tắc khớp shape: so từ phải sang, mỗi chiều phải **bằng nhau hoặc một bên là 1**.
`(4,3) + (3,)` khớp; `(4,3) + (4,)` thì không — thử mà xem (bỏ comment):

In [ ]:
# Bỏ comment để xem lỗi shape mismatch "kinh điển":
# gia_4tp_3thang + np.array([1, 2, 3, 4])
print("(4,3) + (4,) sẽ báo: operands could not be broadcast together")

## 5. Thống kê theo trục, NaN và phân vị

In [ ]:
print("TB mỗi tháng    (axis=0):", gia_4tp_3thang.mean(axis=0))
print("TB mỗi thành phố (axis=1):", gia_4tp_3thang.mean(axis=1).round(1))
# Ghi nhớ: axis nào bị gọi tên, chiều đó biến mất khỏi kết quả

In [ ]:
# argmax: KHÔNG trả giá trị lớn nhất, mà trả VỊ TRÍ của nó
thanh_pho = np.array(["Hà Nội", "Đà Nẵng", "Huế", "TP.HCM"])
tong_gia = gia_4tp_3thang.sum(axis=1)
print("Tổng theo thành phố:", tong_gia)
print("Đắt nhất:", thanh_pho[tong_gia.argmax()])

In [ ]:
# NaN: mean() "lây" NaN để báo động; họ nan* chủ động bỏ qua
gia_thieu = np.array([1200, np.nan, 950])
print(gia_thieu.mean())          # nan — có ít nhất 1 giá trị thiếu!
print(np.nanmean(gia_thieu))     # 1075.0 — TÔI QUYẾT ĐỊNH bỏ qua NaN

In [ ]:
# phân vị (percentile): thước đo không sợ ngoại lai — nền của quy tắc QA buổi 10
gia_lech_phai = rng.lognormal(7, 0.5, size=10_000)
p50, p95, p99 = np.percentile(gia_lech_phai, [50, 95, 99])
print(f"P50 = {p50:,.0f} | P95 = {p95:,.0f} | P99 = {p99:,.0f}")
print(f"mean = {gia_lech_phai.mean():,.0f}  (> P50 vì đuôi phải kéo lên)")

## 6. Cạm bẫy: lát cắt là VIEW

In [ ]:
M = gia_4tp_3thang.copy()      # làm bản riêng để nghịch
hang_dau = M[0]                # VIEW — nhìn chung khối nhớ với M
hang_dau[0] = 999
print("M[0] =", M[0], "  <-- bị sửa theo!")

hang_doc_lap = M[1].copy()     # bản sao thật
hang_doc_lap[0] = -1
print("M[1] =", M[1], " | bản sao:", hang_doc_lap)

## 7. Bài tập tại lớp

### Bài 1 — Đếm ngoại lai (outlier) bằng z-score

Chuẩn hoá mảng giá về z-score ( \(z = (x - \bar{x})/s\) ) rồi đếm số phần tử có \(|z| > 2\).
Tất cả **không dùng vòng for**.

In [ ]:
gia_bt = rng.normal(1000, 200, size=1_000)
gia_bt[::100] = 5000              # cấy 10 ngoại lai vào các vị trí 0,100,200,...

# TODO: tính z rồi đếm |z| > 2 (gợi ý: np.abs)
z = (gia_bt - gia_bt.mean()) / gia_bt.std()
so_outlier = (np.abs(z) > 2).sum()
print("Số outlier:", so_outlier)
assert so_outlier >= 10, "phải bắt được ít nhất 10 outlier đã cấy!"

### Bài 2 — Ma trận doanh thu

`doanh_thu` là ma trận 4 thành phố × 12 tháng. Trả lời bằng code (mỗi câu ≤ 2 dòng):

1. Tháng cao điểm (chỉ số 0–11) của **từng** thành phố.
2. Thành phố có tổng doanh thu năm lớn nhất (in **tên**).
3. Tỷ lệ số ô (thành phố, tháng) vượt 120.

In [ ]:
ten_tp = np.array(["Hà Nội", "Đà Nẵng", "Huế", "TP.HCM"])
doanh_thu = rng.normal(100, 25, size=(4, 12)).round()

# TODO: 3 câu, mỗi câu 1-2 dòng
print("1) Tháng cao điểm:", doanh_thu.argmax(axis=1))
print("2) Vô địch:", ten_tp[doanh_thu.sum(axis=1).argmax()])
print("3) Tỷ lệ ô > 120:", (doanh_thu > 120).mean().round(3))

### Bài 3 — Gán nhãn 3 mức bằng `np.select`

`np.where` chỉ if-else 2 nhánh. Ba nhánh trở lên dùng `np.select(conditions, choices, default)`.
Gán nhãn giá: `"rẻ"` (< 800), `"vừa"` (800–1500), `"đắt"` (> 1500).

In [ ]:
gia_bt3 = np.array([500, 900, 2000, 1200, 700, 1600])

# TODO: hoàn thiện conditions/choices
nhan = np.select(
    [gia_bt3 < 800, gia_bt3 <= 1500],
    ["rẻ", "vừa"],
    default="đắt",
)
list(zip(gia_bt3, nhan))

## 8. Bài tập về nhà — Monte Carlo một khách sạn mini

Mô phỏng 10.000 "năm kinh doanh" của một homestay, **không viết vòng for nào**:

- Mỗi ngày có số khách ~ `rng.poisson(2)`; giá mỗi khách ~ `rng.normal(500, 100)` (nghìn đồng).
- Doanh thu năm = tổng của 365 ngày. Sinh ma trận `(10_000, 365)`.
- Trả lời: doanh thu năm trung bình? P5–P95 (khoảng "năm xui – năm hên")?
  Xác suất một năm thu > 400 triệu?

Gợi ý: `rng.poisson(2, size=(10_000, 365))`, nhân với ma trận giá cùng shape, `sum(axis=1)`,
`np.percentile`, mask + `mean()`.

In [ ]:
RUN_CHALLENGE = False   # đổi True rồi viết code của bạn

if RUN_CHALLENGE:
    kh = rng.poisson(2, size=(10_000, 365))
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| Mảng cùng dtype, tính trong C | Nhanh hơn vòng for hàng chục lần |
| `sum`/`mean` trên bool = đếm/tỷ lệ | dạng KPI đếm/tỷ lệ gặp suốt từ đây về sau |
| `axis` nào gọi tên, chiều đó biến mất | Đọc đúng mọi phép tổng hợp ma trận |
| NaN lây; `nan*` là lựa chọn chủ động | Không để "quy ước ngầm" quyết định kết quả |
| Lát cắt là view — `.copy()` khi cần độc lập | Tránh bug "dữ liệu tự đổi" ở cả NumPy lẫn pandas |

**Buổi sau:** pandas — mảng NumPy có **nhãn** (tên cột, index) thành bảng dữ liệu thật.